## Transformacion de Bronze a Silver


In [0]:
from pyspark.sql.functions import col, regexp_replace, trim, lower, when, try_to_date, to_timestamp, coalesce, lit, upper, try_to_date

# localizaciones de las capas bronze y silver
bronze = "workspace.ferreteria_bronze"
silver = "workspace.ferreteria_silver"

# ===========================
# Limpieza de tabla productos
# 1. En precio, precio + iva y precio venta, se elimina el simbolo de dolar, los puntos
# Se convierte en int.
df_productos = spark.table(f"{bronze}.bronze_productos") \
    .withColumn("Precio", regexp_replace(col("Precio"), r'[\$\.\s]', '').cast("int")) \
    .withColumnRenamed("Precio", "costo") \
    .withColumn("Precio + IVA", regexp_replace(col("Precio + IVA"), r'[\$\.\s]', '').cast("int")) \
    .withColumnRenamed("Precio + IVA", "costo_iva") \
    .withColumn("Precio venta", regexp_replace(col("Precio venta"), r'[\$\.\s]', '').cast("int")) \
    .withColumnRenamed("Precio venta", "precio_venta") \
    .withColumnRenamed("Nombre del producto", "nombre") \
    .withColumnRenamed("Formato de venta", "formato_venta") \
    .withColumnRenamed("Código", "id") \
    .withColumnRenamed("Embalaje", "embalaje") \
    .withColumnRenamed("Categoría", "categoria") \
    .withColumnRenamed("Subcategoría", "subcategoria") \

df_productos.write.format("delta").mode("overwrite").saveAsTable(f"{silver}.productos")


# ============================
# Limpieza de tabla vendedores
# 1. En nombre y apellido, se eliminan los espacios y se pasan a mayusculas
# 2. El rut queda en formato XXXXXXXX-X
# 3. En sueldo se elimina el punto.
# 4. En genero, si es hombre o mujer, se cambia a masculino y femenino respectivamente.
# 5. En activo, si es si o no, se cambia a true o false respectivamente.
# 6. En fecha_nacimiento y fecha_contratacion, se arregla el formato, con to_date.
# 7. Direccion vivienda, tipo contrato y jornada, se eliminan los espacios y se pasan a minusculas.
df_vendedores = spark.table(f"{bronze}.bronze_vendedores")

df_vendedores = df_vendedores.toDF(*[c.strip() for c in df_vendedores.columns]) \
    .withColumnRenamed(" fecha_nacimiento", "fecha_nacimiento") \
    .withColumn("nombre", lower(trim(col("nombre")))) \
    .withColumn("apellido", lower(trim(col("apellido")))) \
    .withColumn("rut", 
        when((col("rut").isNull()) | (trim(col("rut")) == ""), lit("No Registrado"))
        .otherwise(
            regexp_replace(
                regexp_replace(upper(trim(col("rut"))), r'[^0-9K]', ''), 
                r'(.+)(.)$',
                r'$1-$2'
            )
        )
    ) \
    .withColumn("sueldo", regexp_replace(col("sueldo"), r'\.', '').cast("int")) \
    .withColumn("genero", lower(trim(col("genero")))) \
    .withColumn("genero", when(col("genero") == "hombre", "masculino")
                          .when(col("genero") == "mujer", "femenino")
                          .otherwise(col("genero"))) \
    .withColumn("activo", when(lower(trim(col("activo"))) == "si", True).otherwise(False)) \
    .withColumn("fecha_nacimiento", coalesce(
        try_to_date(trim(col("fecha_nacimiento")), "dd-MM-yyyy"),
        try_to_date(trim(col("fecha_nacimiento")), "dd/MM/yyyy"),
        try_to_date(trim(col("fecha_nacimiento")), "dd/MM/yy"),
        try_to_date(trim(col("fecha_nacimiento")), "dd-MM-yy")
    )) \
    .withColumn("correo", lower(trim(col("correo")))) \
    .withColumn("fecha_contratacion", coalesce(
        try_to_date(trim(col("fecha_contratacion")), "dd-MM-yyyy"),
        try_to_date(trim(col("fecha_contratacion")), "dd/MM/yyyy"),
        try_to_date(trim(col("fecha_contratacion")), "dd/MM/yy"),
        try_to_date(trim(col("fecha_contratacion")), "dd-MM-yy")
    )) \
    .withColumn("direccion_vivienda", lower(trim(col("direccion_vivienda")))) \
    .withColumn("direccion_vivienda", when((col("direccion_vivienda").isNull()) | (col("direccion_vivienda") == ""), lit("No Registrado")) 
        .otherwise(col("direccion_vivienda"))) \
    .withColumn("tipo_contrato", lower(trim(col("tipo_contrato")))) \
    .withColumn("jornada", lower(trim(col("jornada")))) \

df_vendedores.write.format("delta").mode("overwrite").saveAsTable(f"{silver}.vendedores")


# ==========================
# Limpieza de tabla clientes
# 1. La columna genero, se pasa a minusculas, se elimina el espacio en blanco y se cambia a masculino o femenino respectimamente.
# 2. La columna membresia se le eliminan los espacios y se cambia a minusculas.
# 3. La columna correo se le quitan los espacios, se pasa a minuscula y luego, si es nulo o vacio, se le asigna el valor de no registrado.
# 4. En fecha_nacimiento y fecha_inscripcion, se arregla el formato, con to_date.
# 5. Se formatea correctamente el numero de telefono.
df_clientes = spark.table(f"{bronze}.bronze_clientes") \
    .withColumn("rut", 
        when((col("rut").isNull()) | (trim(col("rut")) == ""), lit("No Registrado"))
        .otherwise(
            regexp_replace(
                regexp_replace(upper(trim(col("rut"))), r'[^0-9K]', ''), 
                r'(.+)(.)$',
                r'$1-$2'
            )
        )
    ) \
    .withColumn("nombre", lower(trim(col("nombre")))) \
    .withColumn("apellido", lower(trim(col("apellido")))) \
    .withColumn("genero", lower(trim(col("genero")))) \
    .withColumn("genero", when(col("genero") == "hombre", "masculino")
                          .when(col("genero") == "mujer", "femenino")
                          .otherwise(col("genero"))) \
    .withColumn("membresia", lower(trim(col("membresia")))) \
    .withColumn("correo", lower(trim(col("correo")))) \
    .withColumn("correo", when((col("correo").isNull()) | (col("correo") == ""), lit("No Registrado"))
                      .otherwise(col("correo")))\
    .withColumn("fecha_nacimiento", coalesce(
        try_to_date(col("fecha_nacimiento"), "dd-MM-yyyy"),
        try_to_date(col("fecha_nacimiento"), "dd/MM/yyyy"),
        try_to_date(col("fecha_nacimiento"), "dd/MM/yy"),
        try_to_date(col("fecha_nacimiento"), "dd-MM-yy")
    )) \
    .withColumn("fecha_inscripcion", coalesce(
        try_to_date(col("fecha_inscripcion"), "dd-MM-yyyy"),
        try_to_date(col("fecha_inscripcion"), "dd/MM/yyyy"),
        try_to_date(col("fecha_inscripcion"), "dd/MM/yy"),
        try_to_date(col("fecha_inscripcion"), "dd-MM-yy")
    )) \
    .withColumn("sector vivienda",
        when((col("sector vivienda").isNull()) | (col("sector vivienda") == ""), lit("No Registrada")).otherwise(col("sector vivienda"))) \
    .withColumnRenamed("sector vivienda", "sector_vivienda") \
    .withColumn("telefono", 
        when((col("telefono").isNull()) | (trim(col("telefono")) == ""), lit("No Registrado"))
        .otherwise(
            regexp_replace(
                regexp_replace(col("telefono"), r'[^0-9]', ''), 
                r'.*([0-9]{8})$', 
                r'9$1'
            )
        )
    ) \

df_clientes.write.format("delta").mode("overwrite").saveAsTable(f"{silver}.clientes")


# ========================
# Limpieza de tabla ventas
# 1. Al venir en un formato correcto, la columna fecha se pasa a timestamp.
# 2. La columna total se pasa a int
# 3. La columna factura se pasa a booleano.
df_ventas = spark.table(f"{bronze}.bronze_ventas") \
    .withColumn("fecha", to_timestamp(col("fecha"))) \
    .withColumn("total", col("total").cast("int")) \
    .withColumn("factura", when(lower(trim(col("factura"))) == "si", True).otherwise(False)) \

df_ventas.write.format("delta").mode("overwrite").saveAsTable(f"{silver}.ventas")


# ================================
# Limpieza de tabla detalle_ventas
# 1. Las columnas id_venta, id_producto, cantidad y precio unitario se pasan a int
# 2. La columna total se pasa a int
# 3. La columna factura se pasa a booleano.
df_detalle = spark.table(f"{bronze}.bronze_detalle_ventas") \
    .withColumn("id_venta", col("id_venta").cast("int")) \
    .withColumn("id_producto", col("id_producto").cast("int")) \
    .withColumn("cantidad", col("cantidad").cast("int")) \
    .withColumn("precio_unitario", col("precio_unitario").cast("int"))

df_detalle.write.format("delta").mode("overwrite").saveAsTable(f"{silver}.detalle_ventas")


## Ejecutar SOLO si rehice los CSV

In [0]:
%sql
DROP TABLE IF EXISTS workspace.ferreteria_bronze.bronze_ventas;
DROP TABLE IF EXISTS workspace.ferreteria_bronze.bronze_clientes;
DROP TABLE IF EXISTS workspace.ferreteria_bronze.bronze_detalle_ventas;

## Eliminar silvers

In [0]:
%sql
DROP TABLE IF EXISTS workspace.ferreteria_silver.productos;
DROP TABLE IF EXISTS workspace.ferreteria_silver.vendedores;
DROP TABLE IF EXISTS workspace.ferreteria_silver.clientes;
DROP TABLE IF EXISTS workspace.ferreteria_silver.ventas;
DROP TABLE IF EXISTS workspace.ferreteria_silver.detalle_ventas;

In [0]:
%sql
SELECT * from workspace.ferreteria_silver.clientes

id,rut,nombre,apellido,genero,fecha_nacimiento,correo,telefono,fecha_inscripcion,membresia,sector_vivienda
1,10942450-3,juan,cortés,masculino,1996-09-10,juangmail.com,932725604,2022-01-20,basica,fundo el carmen
2,13902046-0,alfredo,diaz,masculino,1995-10-16,alfredo@gmailcom,976203322,2022-01-25,basica,los conquistadores
3,15733993-5,maría,caldera,femenino,2000-01-19,No Registrado,926323261,2024-02-19,pro,No Registrada
4,16605866-2,constanza,diaz,femenino,1978-07-27,constanza@gmail.com,903101144,2021-12-09,basica,los conquistadores
5,12638517-9,paula,díaz,femenino,1990-02-26,No Registrado,918565781,2023-04-02,basica,temuco
6,21555951-6,gonzalo,razo,masculino,1980-06-13,No Registrado,No Registrado,2021-01-05,basica,barrio ingles
7,10880418-2,bernardo,gimenez,masculino,1991-10-05,bernardo9@gmail.com,934380913,2023-12-20,basica,padre las casas
8,16451205-7,sara,cintrón,femenino,1967-03-19,sara@gmail.com,909344364,2021-10-09,pro,barrio ingles
9,14766719-4,constanza,núñez,femenino,1993-01-16,constanza5@gmail.com,928773231,2023-01-22,pro,padre las casas
10,19298503-3,maría,hurtado,femenino,1993-12-29,maría@gmailcom,978693582,2022-01-07,pro,padre las casas


In [0]:
%sql
SELECT * from workspace.ferreteria_silver.productos

nombre,id,costo,costo_iva,formato_venta,embalaje,precio_venta,categoria,subcategoria
SPRAY NEGRO OPACO,4,1299,1546,Pack 6 un,6 un,3680,Pinturas,Aerosoles
SPRAY ROJO VIVO,6,1299,1546,Pack 6 un,6 un,3680,Pinturas,Aerosoles
SPRAY VERDE ESMERALDA,77,1299,1546,Pack 6 un,6 un,3680,Pinturas,Aerosoles
SPRAY AZUL CIELO,15,1299,1546,Pack 6 un,6 un,3680,Pinturas,Aerosoles
SPRAY GRIS MAQUINA,22,1299,1546,Pack 6 un,6 un,3680,Pinturas,Aerosoles
SPRAY BERMELLON,23,1299,1546,Pack 6 un,6 un,3680,Pinturas,Aerosoles
SPRAY AMARILLO REY,25,1299,1546,Pack 6 un,6 un,3680,Pinturas,Aerosoles
SPRAY AZUL PACIFICO,133,1299,1546,Pack 6 un,6 un,3680,Pinturas,Aerosoles
SPRAY LACA BRILLANTE,190,1299,1546,Pack 6 un,6 un,3680,Pinturas,Aerosoles
SPRAY NARANJO,134,1299,1546,Pack 6 un,6 un,3680,Pinturas,Aerosoles


In [0]:
%sql
SELECT * from workspace.ferreteria_silver.detalle_ventas

id_venta,id_producto,cantidad,precio_unitario
1,60294,1,107590
1,60232,14,2510
1,18102,14,4760
1,17360,2,14490
1,10267,4,1580
1,10606,1,1480
1,60229,5,37670
1,10351,3,2400
1,60302,6,16250
1,18184,15,15720


In [0]:
%sql
SELECT * from workspace.ferreteria_silver.vendedores

id,rut,nombre,apellido,genero,fecha_nacimiento,correo,telefono,direccion_vivienda,fecha_contratacion,tipo_contrato,activo,sueldo,jornada
1,21606429-1,david,baez,masculino,2004-04-03,david@gmail.com,111111111,av javiera carrera 045,2025-05-02,indefinido,true,100000,completa
2,11111111-1,maria,gonzalez,femenino,1995-08-15,maria.gonzalez@gmail.com,987654321,calle san martin 789,2024-01-10,indefinido,true,120000,completa
3,17892345-2,carlos,muñoz,masculino,2092-11-22,carlos.munoz@gmail.com,976543210,av alemania 1020,2023-03-15,plazo fijo,true,110000,completa
4,20345678-9,ana,rojas,femenino,2001-05-05,ana.rojas@gmail.com,965432109,las heras 455,2025-06-01,indefinido,true,80000,media
5,18567123-K,jose,pinto,masculino,2094-09-30,jose.pinto@gmail.com,954321098,No Registrado,2022-12-12,indefinido,false,100000,completa
6,19456789-2,camila,soto,femenino,1998-02-14,camila.soto@gmail.com,943210987,av prat 567,2024-04-20,indefinido,true,95000,completa
7,15890123-K,luis,tapia,masculino,2090-07-22,luis@gmail.com,932109876,calle bulnes 890,2022-10-15,indefinido,true,110000,completa
8,17654321-4,francisca,silva,femenino,2002-11-05,francisca.silva@gmail.com,921098765,vicuña mackenna 432,2025-02-02,plazo fijo,true,85000,media
9,21345678-9,pedro,morales,masculino,2095-10-12,pedro.morales@gmail.com,910987654,av alemania 1500,2023-08-18,indefinido,true,105000,completa


In [0]:
%sql
SELECT * from workspace.ferreteria_silver.ventas

id,id_cliente,id_vendedor,tienda,fecha,metodo_pago,factura,descuento,total
1,0,2,Av Javierra Carrera 045,2024-03-04T13:53:41.000Z,credito,false,0,775000
2,53,6,Pedro Salinas 0295,2023-04-12T09:35:27.000Z,transferencia,false,15,3435640
3,0,5,Pedro Salinas 0295,2020-05-12T10:04:42.000Z,debito,false,0,2079600
4,0,4,Pedro Salinas 0295,2021-06-19T17:41:24.000Z,transferencia,false,0,263420
5,59,5,Pedro Salinas 0295,2024-07-07T10:59:43.000Z,debito,false,15,2037059
6,0,2,Pedro Salinas 0295,2022-08-03T12:21:37.000Z,debito,false,0,1181490
7,0,8,Pedro Salinas 0295,2025-08-18T16:47:56.000Z,efectivo,false,0,180610
8,0,6,Hayden 102,2024-04-15T18:42:39.000Z,credito,true,0,2665810
9,0,6,Hayden 102,2021-12-31T11:14:38.000Z,efectivo,false,0,119640
10,10,3,Pedro Salinas 0295,2023-12-11T17:29:19.000Z,debito,false,15,157276
